# Exercise 4 - RAG: retrieve, then generate (slides 18-25)

This notebook runs **inside IRIS**: its Python kernel is IRIS's own Python, so `import iris` talks to the database
directly - no driver, no connection string. The SQL below is the same SQL you ran in the Management Portal.

**Connect the notebook (once):** in VS Code, click **Select Kernel** (top right) -> **Select Another Kernel...** ->
**Existing Jupyter Server...** -> paste `http://127.0.0.1:48888/?token=workshop` -> pick **Python 3 (IRIS)**.

Run the cells top to bottom with **Shift+Enter**. Two cells contain a TODO (10 and 11): the same TODOs as
`Workshop.Exercise.Ex4` - do them here in Python, or in ObjectScript (last section), whichever you prefer.
Stuck? The completed notebook is `solutions/ex4_rag.ipynb`.

In [ ]:
# Step 1: paste the workshop's OpenAI key (shared, valid today only) between the quotes, then run this cell.
# It is stored in IRIS (Workshop.Util.SetKey) - the ObjectScript option uses the same key. Run once; you can clear it afterwards.
OPENAI_KEY = ""

import json, html
import iris
from IPython.display import HTML, display

util = iris.cls("Workshop.Util")
if OPENAI_KEY:
    print(util.SetKey(OPENAI_KEY))
# Ready check: model downloaded (Exercise 1), chunks from your Exercise 2, the key.
SYSTEM_PROMPT = util.SystemPrompt()    # data/prompts/system_prompt.txt
print(util.Checkpoint(4))      # catches up if needed: model + registration (Ex 1), chunks with YOUR Ex 2 code
print("LLM:", util.LlmStatus())

## 1. Retrieve: top-k chunks with SQL

The question becomes a vector with the **same model** as the chunks (`EMBEDDING(?)`), then
`VECTOR_DOT_PRODUCT` ranks the chunks (Exercise 1). This is the query from Exercise 2, with the question as a parameter.

In [ ]:
SEARCH_SQL = '''
SELECT TOP ? chunk_id, title, source, content,
       VECTOR_DOT_PRODUCT(embedding, EMBEDDING(?)) AS score
FROM Workshop.SupportChunks
ORDER BY score DESC'''

def search(question, k=3, min_score=None):
    rows = iris.sql.exec(SEARCH_SQL, k, question)
    hits = [{"chunk_id": r[0], "title": r[1], "source": r[2], "content": r[3], "score": float(r[4])} for r in rows]
    if min_score is not None:
        hits = [h for h in hits if h["score"] >= min_score]
    return hits

for h in search("How long do we keep lab backups?"):
    print(f"{h['score']:.3f}  {h['chunk_id']:<10} {h['title']}")

## 2. Build the context (TODO 4)

The LLM gets the retrieved passages as text. Each block starts with the chunk ID in brackets - that is what the
LLM will cite, so we can check every citation against what was retrieved:

```
[backup-restore §2] Backup schedule and retention (source: backup-restore-handbook.pdf)
passage text

[backup-restore §1] ...
```

In [ ]:
def build_context(hits):
    blocks = [f"[{h['chunk_id']}] {h['title']} (source: {h['source']})\n{h['content']}" for h in hits]
    return "\n\n".join(blocks)

print(build_context(search("How long do we keep lab backups?")))

`show()` displays a result: the answer next to its evidence, cited passages marked. Run it once - the next sections (and the
ObjectScript option in section 7) use it.

In [ ]:
def show(result, show_prompt=False):
    """The answer next to its evidence: cited passages are marked."""
    if "error" in result:
        display(HTML(f"<b style='color:#b00'>Error:</b> {html.escape(result['error'])}"))
        return
    r = result["reply"]
    cited = set(r.get("cited_chunk_ids", []))
    color = "#0a7d32" if r.get("sufficient_evidence") else "#a15c00"
    rows = "".join(
        f"<tr><td>{'&#10004;' if h['chunk_id'] in cited else ''}</td><td>{h['score']:.3f}</td>"
        f"<td><code>{html.escape(h['chunk_id'])}</code></td><td>{html.escape(h['content'][:160])}...</td></tr>"
        for h in result["hits"])
    warn = (f"<p style='color:#b00'>Cited but not retrieved: {result['unknown_citations']}</p>"
            if result["unknown_citations"] else "")
    page = (f"<p><b>Q:</b> {html.escape(result['question'])}</p>"
            f"<p style='border-left:4px solid {color};padding-left:8px'><b>A:</b> {html.escape(r.get('answer', ''))}</p>"
            f"<p style='color:{color}'>sufficient_evidence = {r.get('sufficient_evidence')} &nbsp; "
            f"<small>({r.get('model', '')}, {r.get('latency_ms', 0):.0f} ms)</small></p>{warn}"
            f"<table><tr><th>cited</th><th>score</th><th>chunk</th><th>passage</th></tr>{rows}</table>")
    if show_prompt:
        page += f"<details open><summary>Prompt sent to the LLM</summary><pre>{html.escape(result['prompt'])}</pre></details>"
    display(HTML(page))

## 3. Ask: retrieve, then generate (TODO 5)

The system prompt (`data/prompts/system_prompt.txt`) tells the LLM to answer **only** from the passages and to
cite chunk IDs. The LLM call itself is IRIS code (`Workshop.Util.CompleteJSON`, an HTTPS request from IRIS to
OpenAI with the key you pasted in step 1): you give it the system prompt and your message, it returns JSON text
`{"sufficient_evidence": ..., "answer": ..., "cited_chunk_ids": [...]}` (or `{"error": ...}`).

In [ ]:
def ask(question, k=3, min_score=None):
    hits = search(question, k, min_score)
    context = build_context(hits) or "(no passages retrieved)"
    user_msg = "Passages:\n\n" + context + "\n\nQuestion: " + question
    reply = json.loads(util.CompleteJSON(SYSTEM_PROMPT, user_msg))
    if "error" in reply:
        raise RuntimeError(reply["error"])
    retrieved = {h["chunk_id"] for h in hits}
    unknown = [c for c in reply.get("cited_chunk_ids", []) if c not in retrieved]
    return {"question": question, "hits": hits, "prompt": user_msg, "reply": reply, "unknown_citations": unknown}

show(ask("How long do we keep lab backups?"))

## 4. Test it: read each answer next to its evidence

* a question the documents answer, asked in other words (paraphrase),
* a question the documents do **not** answer - a good RAG system says so instead of guessing.

In [ ]:
paraphrase = ask("For how many days are the nightly snapshots of a lab environment retained?")
unsupported = ask("What uptime percentage does the platform guarantee in its SLA?")
show(paraphrase)
show(unsupported)

## 5. Add knowledge without retraining: the runbook (failed-imports §1)

Ask, add one PDF to the knowledge base (with **your** Exercise 2 code), ask again. Nothing about the model changes -
only the data in IRIS. At the end the runbook is removed again, so you can repeat this.

In [ ]:
question = "Where do failed imports go in our environment, and how long are they kept?"
before = ask(question)
show(before)
print("ingested chunks:", util.IngestRunbook())
try:
    after = ask(question)
    show(after)
finally:
    util.RemoveRunbook()             # always: the next run starts without the runbook again
    print("runbook removed again")

In [ ]:
# Success check (same checks as Workshop.Exercise.Ex4.Run)
a, u, b, f = ask("How long do we keep lab backups?")["reply"], unsupported["reply"], before["reply"], after["reply"]
assert "[backup-restore §2]" in build_context(search("How long do we keep lab backups?")), "build_context should contain [chunk_id] markers"
assert a["sufficient_evidence"] and "14" in a["answer"], "answerable question should say 14 days"
assert not u["sufficient_evidence"] and not u["cited_chunk_ids"], "unsupported question should cite nothing"
assert not b["sufficient_evidence"], "before ingestion there is no evidence"
assert "failed-imports §1" in f["cited_chunk_ids"] and "11" in f["answer"], "after ingestion: cite failed-imports §1 and say 11 days"
assert not after["unknown_citations"], "citations must be retrieved chunks"
print("EXERCISE 4 PASSED")

## 6. Experiment

Change the settings and run the cell again:

* `K` - how many passages the LLM gets. Try 1 and 8.
* `MIN_SCORE` - drop weak passages (e.g. `0.3`). What happens to the SLA question?
* `SHOW_PROMPT` - see exactly what is sent to the LLM.

In [ ]:
QUESTION = "My password reset email never showed up. What should I do?"
K = 3
MIN_SCORE = None        # e.g. 0.3
SHOW_PROMPT = True

show(ask(QUESTION, K, MIN_SCORE), show_prompt=SHOW_PROMPT)

## 7. The ObjectScript option

If you did TODO 4 and 11 in `Workshop.Exercise.Ex4` (VS Code) instead, run your ObjectScript code from here.
Same pipeline, same result - Python and ObjectScript call the same IRIS.

In [ ]:
def ask_objectscript(question, k=3, min_score=None):
    res = json.loads(iris.cls("Workshop.Exercise.Ex4").AskJSON(question, k, "" if min_score is None else min_score))
    if "error" in res:
        return res
    return {"question": question, "hits": res["retrieval"]["hits"], "prompt": "(built in ObjectScript)",
            "reply": res["reply"], "unknown_citations": res["unknown_citations"]}

show(ask_objectscript("How long do we keep lab backups?"))